In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.nn.functional import relu
from torch.utils.data import DataLoader, Dataset
import unet

import matplotlib.pyplot as plt

In [2]:
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print(f"Working on {device}")

Working on xpu


# Dataset

In [3]:
class TestDataset(Dataset):
    def __init__(self, train, transform=None):
        self.n_elements=4
        self.transform = transform

        if train:
          self.images = torch.rand((self.n_elements, 1, 512, 512))
          self.masks = torch.rand((self.n_elements, 2, 512, 512))
        else:
          self.images = torch.rand((self.n_elements, 1, 512, 512))
          self.masks = torch.rand((self.n_elements, 2, 512, 512))

    def __len__(self):
        return self.n_elements

    def __getitem__(self, idx):
        image = self.images[idx]

        if self.transform:
            image = self.transform(image)

        return image, self.masks[idx]

# Interactive part

In [4]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)

    model.train() # set the model to training mode
    for batch, (X,y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        
        # compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # backpropagation
        loss.backward() # backpropagate the prediction loss
        optimizer.step() # adjust the parameters by the gradients collected in the backward pass
        optimizer.zero_grad() # reset the gradients of model parameters

        loss, current = loss.item(), batch * batch_size + len(X)
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn, device):
    model.eval() # set model to evaluation mode
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss = 0
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    
    test_loss /= num_batches
    print(f"Test Error: \n Avg loss: {test_loss:>8f} \n")

In [5]:
learning_rate = 1e-3
batch_size = 2
epochs = 1

# loading data
train_data = TestDataset(train=True)
test_data = TestDataset(train=False)
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# defyning loss function
loss_fn = unet.dice_loss

model = unet.UNet(n_class=2)
model.to(device)

# defyning optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate
)

In [6]:
# training loop
for t in range(epochs):
    print(f"Epoch {t+1}\n------------")
    train_loop(train_dataloader, model, loss_fn, optimizer, device)
    test_loop(test_dataloader, model, loss_fn, device)

Epoch 1
------------
onednn_verbose,v1,info,oneDNN v3.9.1 (commit 80a3a8e745d2f0186e674b0af9332fd6e074c94f)
onednn_verbose,v1,info,cpu,runtime:threadpool,nthr:7
onednn_verbose,v1,info,cpu,isa:Intel AVX2 with Intel DL Boost
onednn_verbose,v1,info,gpu,runtime:DPC++
onednn_verbose,v1,info,gpu,engine,sycl gpu device count:1 
onednn_verbose,v1,info,gpu,engine,0,backend:Level Zero,name:Intel(R) Graphics [0x7d41],driver_version:1.3.30049,binary_kernels:enabled
onednn_verbose,v1,info,graph,backend,0:dnnl_backend
onednn_verbose,v1,primitive,info,template:operation,engine,primitive,implementation,prop_kind,memory_descriptors,attributes,auxiliary,problem_desc,exec_time
onednn_verbose,v1,graph,info,template:operation,engine,partition_id,partition_kind,op_names,data_formats,logical_tensors,fpmath_mode,implementation,backend,exec_time
onednn_verbose,v1,common,error,runtime,Intel(R) Graphics [0x7d41] OpenCL device not found,src/gpu/intel/sycl/utils.cpp:136


RuntimeError: could not create a primitive

In [17]:
X = torch.rand((1,1,512,512)).to(device)
y = torch.rand((1,2,512,512)).to(device)

In [19]:
pred = model(image)
loss = loss_fn(pred, y)

loss.backward()

onednn_verbose,v1,info,oneDNN v3.9.1 (commit 80a3a8e745d2f0186e674b0af9332fd6e074c94f)
onednn_verbose,v1,info,cpu,runtime:threadpool,nthr:7
onednn_verbose,v1,info,cpu,isa:Intel AVX2 with Intel DL Boost
onednn_verbose,v1,info,gpu,runtime:DPC++
onednn_verbose,v1,info,gpu,engine,sycl gpu device count:1 
onednn_verbose,v1,info,gpu,engine,0,backend:Level Zero,name:Intel(R) Graphics [0x7d41],driver_version:1.3.30049,binary_kernels:enabled
onednn_verbose,v1,info,graph,backend,0:dnnl_backend
onednn_verbose,v1,primitive,info,template:operation,engine,primitive,implementation,prop_kind,memory_descriptors,attributes,auxiliary,problem_desc,exec_time
onednn_verbose,v1,graph,info,template:operation,engine,partition_id,partition_kind,op_names,data_formats,logical_tensors,fpmath_mode,implementation,backend,exec_time
onednn_verbose,v1,common,error,runtime,Intel(R) Graphics [0x7d41] OpenCL device not found,src/gpu/intel/sycl/utils.cpp:136


RuntimeError: could not create a primitive

In [3]:
class SampleModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.ConvTranspose2d(1024, 512, 2, stride=2)

    def forward(self, X):
        return self.net(X)

loss_fn = unet.dice_loss

sample_model = SampleModel()
sample_model.to(device)

X = torch.rand((1,1024,16,16)).to(device)
y = torch.rand((1, 512, 32, 32)).to(device)

pred = sample_model(X)
loss = loss_fn(pred, y)

In [4]:
loss.backward()

onednn_verbose,v1,info,oneDNN v3.9.1 (commit 80a3a8e745d2f0186e674b0af9332fd6e074c94f)
onednn_verbose,v1,info,cpu,runtime:threadpool,nthr:7
onednn_verbose,v1,info,cpu,isa:Intel AVX2 with Intel DL Boost
onednn_verbose,v1,info,gpu,runtime:DPC++
onednn_verbose,v1,info,gpu,engine,sycl gpu device count:1 
onednn_verbose,v1,info,gpu,engine,0,backend:Level Zero,name:Intel(R) Graphics [0x7d41],driver_version:1.3.30049,binary_kernels:enabled
onednn_verbose,v1,info,graph,backend,0:dnnl_backend
onednn_verbose,v1,primitive,info,template:operation,engine,primitive,implementation,prop_kind,memory_descriptors,attributes,auxiliary,problem_desc,exec_time
onednn_verbose,v1,graph,info,template:operation,engine,partition_id,partition_kind,op_names,data_formats,logical_tensors,fpmath_mode,implementation,backend,exec_time
onednn_verbose,v1,common,error,runtime,Intel(R) Graphics [0x7d41] OpenCL device not found,src/gpu/intel/sycl/utils.cpp:136


RuntimeError: could not create a primitive